# 3. Patterns: when and where is Barcelona saturated?

**Question:** which days, seasons and barris carry the load, and how full does public transport get?

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from prisa import store, figures
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)
con = store.connect()

In [ ]:
figures.city_visitors(store.read('city_day'));

In [ ]:
cd = store.read('city_day'); cd['day'] = pd.to_datetime(cd['day'])
by = lambda key: (cd.groupby(key)['visitors'].mean() / cd['visitors'].mean()).round(3)
pd.DataFrame({'weekday index': by(cd['day'].dt.day_name())}).T, pd.DataFrame({'month index': by(cd['day'].dt.month)}).T

Which weekday and which season carry the load city-wide, and does every barri follow the same pattern?

In [ ]:
bd = store.read('barri_day'); bd['day'] = pd.to_datetime(bd['day'])
summer = bd[bd['day'].dt.month.isin([7, 8])].groupby('zone_id')['visitors'].mean()
winter = bd[bd['day'].dt.month.isin([1, 2])].groupby('zone_id')['visitors'].mean()
figures.barri_map(summer / winter, store.read('barri_geometry'), 'Summer vs winter visitors', cmap='RdBu_r', label='ratio', center=1.0);

If coastal and monument barris peak in a different season from the city as a whole, a city-wide calendar discounts days when the hotspots are fullest - this is why notebook 6 compares three calendar designs.

## How full does public transport get?

`prisa.saturation.transit_load` estimates, per barri and day: rides starting there (trips out x segment mix x mode share from EMEF 2024 and tourist surveys) against places offered by the departures from its stops (GTFS timetables x vehicle capacity at 4 or 6 people/m2), for the busiest hour. One scale factor for riders already on board is calibrated so the FGC weekday peak matches FGC's own measured occupancy (our Geotren archive).

In [ ]:
import json
print(json.loads((store.PROCESSED / 'transit_load_calibration.json').read_text()))
tl = store.read('transit_load')
tl.groupby('standing_density')['ratio_peak_hour'].describe().round(3)

In [ ]:
t4 = tl[tl['standing_density'] == 4.0]
figures.barri_map(t4.groupby('zone_id')['ratio_peak_hour'].median(), store.read('barri_geometry'),
                  'Median peak-hour load (4 people/m2)', cmap='Reds', label='load ratio');